# LAB 5a:  Vertex AI での Keras モデルのトレーニング

**学習目標**

1. Vertex AI Training Service への送信に適した Keras モデルコードを構築する
1. `gcloud ai custom-jobs create` を使用してトレーニング ジョブを Vertex AI に送信する
1. Vertex AI を使用してハイパーパラメータ調整ジョブを構成し、実行する


## はじめに 
このノートブックでは、以前に開発した Wide & Deep モデルをパッケージ化し、Google Cloud の Vertex AI Training Service でモデルをトレーニングします。さらに、Vertex AI のハイパーパラメータ調整を使用してモデルを最適化します。

各学習目標は、この演習用ラボ ノートブックで完了する一連のステップに対応しています。

## 環境変数の設定と必要なライブラリの読み込み

必要なライブラリをインポートしましょう。特に、ハイパーパラメータ調整ジョブからのメトリックを報告するために使用する `cloudml-hypertune` ライブラリをインポートします。

In [ ]:
import os

import hypertune

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

### 環境変数を設定します。

以下で、環境変数 `BUCKET` をご自身の Google Cloud Storage バケットに設定します。バケット名は世界中で一意である必要があり、プロジェクト ID が含まれているのが一般的です。以下の `<BUCKET>` を一意のバケット名に置き換えてください。

In [ ]:
PROJECT = !gcloud config get-value project
PROJECT = PROJECT[0]
BUCKET = PROJECT
REGION = "us-central1"

os.environ["PROJECT"] = PROJECT
os.environ["BUCKET"] = BUCKET
os.environ["REGION"] = REGION

バケットが存在しない場合は作成し、バケットがリージョン対応（regional）であり、Vertex AI の実行リージョンと一致していることを以下で確認します。

In [ ]:
 %%bash
if ! gcloud storage ls | grep -q gs://${BUCKET}/; then
    gcloud storage buckets create --location=${REGION} gs://${BUCKET}
fi
gcloud storage ls -L -b gs://$BUCKET | grep "gs://\|Location"
echo $REGION

In [ ]:
%%bash
gcloud config set project ${PROJECT}
gcloud config set ai/region ${REGION}

## データが存在することの確認

バケット内にデータが存在することを確認します。データが存在しない場合は、ラボ [4a_sample_babyweight](../solutions/4a_sample_babyweight.ipynb) に戻ってデータを作成してください。

In [ ]:
%%bash
gcloud storage ls gs://${BUCKET}/babyweight/data/*000000000000.csv

[Keras Wide & Deep のコード](../solutions/4c_keras_wide_and_deep_babyweight.ipynb) が完成したので、Vertex AI Training Service を使用してモデルをトレーニングできるように、コードを Python パッケージに変換します。

最初のステップは、モデル コード用のディレクトリを作成することです。

In [ ]:
%%bash
mkdir -p babyweight/trainer
touch babyweight/trainer/__init__.py

次に、`%%writefile` マジックを使用して、下のセルの内容を `babyweight/trainer` ディレクトリ内のファイルに書き込みます。

### ハイパーパラメータの引数解析コードを保持するトレーナー モジュールの task.py を作成

In [ ]:
%%writefile babyweight/trainer/task.py
import argparse
import json
import os

from trainer import model

import tensorflow as tf

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--train_data_path",
        help="トレーニング データの GCS の場所",
        required=True
    )
    parser.add_argument(
        "--eval_data_path",
        help="評価データの GCS の場所",
        required=True
    )
    parser.add_argument(
        "--output_dir",
        help="チェックポイントの書き込みおよびモデルのエクスポートを行う GCS の場所",
        default = os.getenv("AIP_MODEL_DIR")
    )
    parser.add_argument(
        "--batch_size",
        help="勾配を計算するサンプル数。",
        type=int,
        default=512
    )
    parser.add_argument(
        "--nnsize",
        help="DNN の隠れ層のサイズ -- スペース区切りのレイヤを指定",
        default="128 32 4"
    )
    parser.add_argument(
        "--nembeds",
        help="主要な実数値パラメータの n 個のクロスに対する埋め込みサイズ",
        type=int,
        default=3
    )
    parser.add_argument(
        "--num_epochs",
        help="モデルをトレーニングするエポック数。",
        type=int,
        default=10
    )
    parser.add_argument(
        "--train_examples",
        help="""トレーニング ジョブを実行するサンプル数（千単位）。
        利用可能な実際のサンプル数を超える場合は、それらを循環（サイクル）します。
        したがって、サンプル数が 100k しかない場合にここで 1000 を指定すると、
        10 エポックになります。""",
        type=int,
        default=5000
    )
    parser.add_argument(
        "--eval_steps",
        help="""モデルを評価する正のステップ数。デフォルトは
        None であり、input_fn が入力終了例外を発生させるまで
        評価することを意味します""",
        type=int,
        default=None
    )

    # すべての引数を解析
    args = parser.parse_args()
    arguments = args.__dict__

    # 一部の引数を変更
    arguments["train_examples"] *= 1000

    # トレーニング ジョブを実行
    model.train_and_evaluate(arguments)

同様に、前のラボで開発したモデルをファイル `model.py` に書き込むことができます。

In [ ]:
%%writefile babyweight/trainer/model.py
import datetime
import os
import shutil
import numpy as np
import tensorflow as tf
import hypertune

# CSV、ラベル、キーの各列を決定
CSV_COLUMNS = [
    "weight_pounds",
    "is_male",
    "mother_age",
    "plurality",
    "gestation_weeks",
]
LABEL_COLUMN = "weight_pounds"
TOTAL_STEPS = 10000

NUMERICAL_COLUMNS = ["mother_age", "gestation_weeks"]
CATEGORICAL_COLUMNS = ["is_male", "plurality"]


# 各 CSV 列のデフォルト値をリストのリストとして設定。
# is_male と plurality は文字列として扱う。
DEFAULTS = [[0.0], ["null"], [0.0], ["null"], [0.0]]


def features_and_labels(row_data):
    """特徴量ディクショナリから特徴量とラベルを分割します。

    Args:
        row_data: CSV 列名とテンソル値のディクショナリ。
    Returns:
        特徴量テンソルのディクショナリとラベル テンソル。
    """
    label = row_data.pop(LABEL_COLUMN)

    return row_data, label  # features, label


def load_dataset(pattern, batch_size=1, mode=tf.estimator.ModeKeys.EVAL):
    """CSV ファイルから tf.data API を使用してデータセットを読み込みます。

    Args:
        pattern: str、ファイル一覧に glob 展開するファイル パターン。
        batch_size: int、バッチあたりのサンプル数。
        mode: トレーニングか評価かを決定する tf.estimator.ModeKeys。
    Returns:
        `Dataset` オブジェクト。
    """
    # CSV データセットを作成
    dataset = tf.data.experimental.make_csv_dataset(
        file_pattern=pattern,
        batch_size=batch_size,
        column_names=CSV_COLUMNS,
        column_defaults=DEFAULTS,
    )

    # データセットを特徴量とラベルにマッピング
    dataset = dataset.map(map_func=features_and_labels)  # features, label

    # トレーニング用にシャッフルしてリピート
    if mode == tf.estimator.ModeKeys.TRAIN:
        dataset = dataset.shuffle(buffer_size=1000).repeat()

    # マルチスレッドを活用。1=AUTOTUNE
    dataset = dataset.prefetch(buffer_size=1)

    return dataset


def create_input_layers():
    """各特徴量の入力レイヤのディクショナリを作成します。

    Returns:
        各特徴量の `tf.Keras.layers.Input` レイヤのディクショナリ。
    """
    deep_inputs = {
        colname: tf.keras.layers.Input(
            name=colname, shape=(1,), dtype="float32"
        )
        for colname in NUMERICAL_COLUMNS
    }

    wide_inputs = {
        colname: tf.keras.layers.Input(name=colname, shape=(1,), dtype="string")
        for colname in CATEGORICAL_COLUMNS
    }

    inputs = {**wide_inputs, **deep_inputs}

    return inputs


def transform(inputs, nembeds):
    """変換された入力のディクショナリを作成します。

    Returns:
        変換された Tensor のディクショナリ
    """

    deep = {}
    wide = {}

    buckets = {
        "mother_age": np.arange(15, 45, 1).tolist(),
        "gestation_weeks": np.arange(17, 47, 1).tolist(),
    }
    bucketized = {}

    for numerical_column in NUMERICAL_COLUMNS:
        deep[numerical_column] = inputs[numerical_column]
        bucketized[numerical_column] = tf.keras.layers.Discretization(buckets[numerical_column])(inputs[numerical_column])
        wide[f"btk_{numerical_column}"] = tf.keras.layers.CategoryEncoding(
            num_tokens=len(buckets[numerical_column]) + 1, output_mode="one_hot"
        )(bucketized[numerical_column])

    crossed = tf.keras.layers.experimental.preprocessing.HashedCrossing(
        num_bins=len(buckets["mother_age"]) * len(buckets["gestation_weeks"])
    )((bucketized["mother_age"], bucketized["gestation_weeks"]))

    deep["age_gestation_embeds"] = tf.keras.layers.Flatten()(
        tf.keras.layers.Embedding(
            input_dim=len(buckets["mother_age"])
            * len(buckets["gestation_weeks"]),
            output_dim=nembeds,
        )(crossed)
    )

    vocab = {
        "is_male": ["True", "False", "Unknown"],
        "plurality": [
            "Single(1)",
            "Twins(2)",
            "Triplets(3)",
            "Quadruplets(4)",
            "Quintuplets(5)",
            "Multiple(2+)",
        ],
    }

    for categorical_column in CATEGORICAL_COLUMNS:
        wide[categorical_column] = tf.keras.layers.StringLookup(
            vocabulary=vocab[categorical_column], output_mode="one_hot"
        )(inputs[categorical_column])

    return wide, deep

def get_model_outputs(wide_inputs, deep_inputs, dnn_hidden_units):
    """モデル アーキテクチャを作成し、出力を返します。

    Args:
        wide_inputs: モデルの Wide 側への入力として使用される Dense テンソル。
        deep_inputs: モデルの Deep 側への入力として使用される Dense テンソル。
        dnn_hidden_units: 長さが隠れ層の数であり、i 番目の要素が i 番目のレイヤのニューロン数である整数のリスト。
    Returns:
        モデルからの Dense テンソル出力。
    """
    # Deep 側の隠れ層
    layers = [int(x) for x in dnn_hidden_units.split()]
    deep = deep_inputs
    for layerno, numnodes in enumerate(layers):
        deep = tf.keras.layers.Dense(
            units=numnodes, activation="relu", name=f"dnn_{layerno + 1}"
        )(deep)
    deep_out = deep

    # Wide 側の線形モデル
    wide_out = tf.keras.layers.Dense(
        units=10, activation="relu", name="linear"
    )(wide_inputs)

    # 2 つの側を連結
    both = tf.keras.layers.Concatenate(name="both")([deep_out, wide_out])

    # 回帰のため最終出力は線形活性化
    output = tf.keras.layers.Dense(units=1, activation="linear", name="weight")(
        both
    )

    return output


def rmse(y_true, y_pred):
    """RMSE 評価指標を計算します。

    Args:
        y_true: tensor、正解ラベル。
        y_pred: tensor、予測ラベル。
    Returns:
        正解ラベルと予測ラベルの間の RMSE の値を持つ Tensor。
    """
    return tf.sqrt(tf.reduce_mean(tf.square(y_pred - y_true)))


def build_wide_deep_model(dnn_hidden_units=[64, 32], nembeds=3):
    """Keras Functional API を使用して Wide & Deep モデルを構築します。

    Returns:
        `tf.keras.models.Model` オブジェクト。
    """
    # 入力レイヤを作成
    inputs = create_input_layers()

    # Wide と Deep の両方について生の特徴量を変換
    wide, deep = transform(inputs, nembeds)

    # Keras の Functional API では次が必要: LayerConstructor()(inputs)
    wide_inputs = tf.keras.layers.Concatenate()(wide.values())
    deep_inputs = tf.keras.layers.Concatenate()(deep.values())

    # 入力を考慮したモデルの出力を取得
    output = get_model_outputs(wide_inputs, deep_inputs, dnn_hidden_units)

    # モデルを構築し、すべてをまとめてコンパイル
    model = tf.keras.models.Model(inputs=inputs, outputs=output)
    model.compile(optimizer="adam", loss="mse", metrics=[rmse, "mse"])

    return model


# HyperTune レポーティング オブジェクトのインスタンス化
hpt = hypertune.HyperTune()

# レポーティング用コールバック
class HPTCallback(tf.keras.callbacks.Callback):

    def on_epoch_end(self, epoch, logs=None):
        global hpt
        hpt.report_hyperparameter_tuning_metric(
            hyperparameter_metric_tag='val_rmse',
            metric_value=logs['val_rmse'],
            global_step=epoch)

        
def train_and_evaluate(args):
    model = build_wide_deep_model(args["nnsize"], args["nembeds"])
    print("Here is our Wide-and-Deep architecture so far:\n")
    print(model.summary())

    trainds = load_dataset(
        args["train_data_path"],
        args["batch_size"],
        tf.estimator.ModeKeys.TRAIN)

    evalds = load_dataset(
        args["eval_data_path"], 1000, tf.estimator.ModeKeys.EVAL)
    if args["eval_steps"]:
        evalds = evalds.take(count=args["eval_steps"])

    num_batches = args["batch_size"] * args["num_epochs"]
    steps_per_epoch = args["train_examples"] // num_batches

    checkpoint_path = os.path.join(args["output_dir"], "checkpoints/babyweight")
    cp_callback = tf.keras.callbacks.ModelCheckpoint(
        filepath=checkpoint_path, verbose=1, save_weights_only=True)

    history = model.fit(
        trainds,
        validation_data=evalds,
        epochs=args["num_epochs"],
        steps_per_epoch=steps_per_epoch,
        verbose=2,  # 0=サイレント, 1=プログレスバー, 2=エポックごとに1行
        callbacks=[cp_callback, HPTCallback()])

    EXPORT_PATH = os.path.join(
        args["output_dir"], datetime.datetime.now().strftime("%Y%m%d%H%M%S"))
    tf.saved_model.save(
        obj=model, export_dir=EXPORT_PATH)  # デフォルトのサービング関数を使用
    
    print("Exported trained model to {}".format(EXPORT_PATH))

## ローカルでトレーニング

コードをパッケージに移行した後、スタンドアロンとして動作することを確認します。パイプラインの開発中にデータセット全体でトレーニングを試みないよう、`--train_examples` フラグを組み込んだことに注目してください。サブセットですべてが動作することを確認したら、すべてのデータでトレーニングできるように変更できます。このサブセットであっても、出力が表示されないまま約 *3 分* かかります...

### トレーナー モジュール パッケージをローカルで実行

単一のファイルに対して、小さなバッチサイズ、1 エポック、1 つのトレーニング サンプル、1 つの評価ステップで非常に小規模なトレーニング ジョブを実行できます。

In [ ]:
%%bash
OUTDIR=babyweight_trained
rm -rf ${OUTDIR}
export PYTHONPATH=${PYTHONPATH}:${PWD}/babyweight
python3 -m trainer.task \
    --train_data_path=gs://${BUCKET}/babyweight/data/train*.csv \
    --eval_data_path=gs://${BUCKET}/babyweight/data/eval*.csv \
    --output_dir=${OUTDIR} \
    --batch_size=10 \
    --num_epochs=1 \
    --train_examples=1 \
    --eval_steps=1

## Vertex AI でのトレーニング

ローカルですべてが動作することを確認できたので、クラウド上でトレーニングを行いましょう！まず、コードをソース ディストリビューションとしてパッケージ化する必要があります。これには `setuptools` を使用できます。

In [ ]:
%%writefile babyweight/setup.py
from setuptools import find_packages
from setuptools import setup

setup(
    name='babyweight_trainer',
    version='0.1',
    packages=find_packages(),
    include_package_data=True,
    description='Babyweight model training application.'
)

In [ ]:
%%bash
cd babyweight
python ./setup.py sdist --formats=gztar
cd ..

パッケージを Cloud Storage バケットに保存します。

In [ ]:
%%bash
gcloud storage cp babyweight/dist/babyweight_trainer-0.1.tar.gz gs://${BUCKET}/babyweight/

Cloud に送信するには [`gcloud custom-jobs create`](https://cloud.google.com/sdk/gcloud/reference/ai/custom-jobs/create) を使用し、Vertex AI Training Service 用のいくつかの追加パラメータを指定するだけです。
- display-name: Cloud ジョブの一意の識別子。通常、一意性を確保するためにシステム時間を付加します
- region: トレーニングを行う Cloud リージョン。サポートされている Vertex AI Training Service のリージョンについては、[こちら](https://cloud.google.com/vertex-ai/docs/general/locations) を参照してください

以前に `gcloud ai custom-jobs create` をコマンドの呼び出し内で直接 `worker pool spec` とパススルーの Python 引数を指定して実行したのを見たことがあるかもしれませんが、ここでは YAML ファイルを使用します。これにより、ハイパーパラメータ調整への移行が容易になります。

`args:` 引数を通じて、`task.py` ファイル用のパススルー引数を追加します。

In [ ]:
%%bash

TIMESTAMP=$(date -u +%Y%m%d_%H%M%S)
OUTDIR=gs://${BUCKET}/babyweight/trained_model_$TIMESTAMP
JOB_NAME=babyweight_$TIMESTAMP

PYTHON_PACKAGE_URI=gs://${BUCKET}/babyweight/babyweight_trainer-0.1.tar.gz
PYTHON_PACKAGE_EXECUTOR_IMAGE_URI="us-docker.pkg.dev/vertex-ai/training/tf-cpu.2-8:latest"
PYTHON_MODULE=trainer.task

echo > ./config.yaml "workerPoolSpecs:
  machineSpec:
    machineType: n1-standard-4
  replicaCount: 1
  pythonPackageSpec:
    executorImageUri: $PYTHON_PACKAGE_EXECUTOR_IMAGE_URI
    packageUris: $PYTHON_PACKAGE_URI
    pythonModule: $PYTHON_MODULE
    args:
    - --train_data_path=gs://${BUCKET}/babyweight/data/train*.csv
    - --eval_data_path=gs://${BUCKET}/babyweight/data/eval*.csv
    - --output_dir=$OUTDIR
    - --num_epochs=10
    - --train_examples=10000
    - --eval_steps=100
    - --batch_size=32
    - --nembeds=8"

gcloud ai custom-jobs create \
  --region=${REGION} \
  --display-name=$JOB_NAME \
  --config=config.yaml

トレーニング ジョブは 10〜15 分以内に完了するはずです。次のラボを完了するには、トレーニング済みのモデルが必要になります。

## ハイパーパラメータ調整

ハイパーパラメータ調整を行うには、YAML ファイルを作成し、その名前を `--config` で渡します。
このステップには <b>数時間</b> かかる場合があります。早く完了させるには、`--parallel-trial-count` を増やすか、`--max-trial-count` を減らします。`--parallel-trial-count` は探索を開始する初期シードの数であるため、あまり大きくしすぎないようにしてください。大きすぎると単なるランダム探索になってしまいます。

In [ ]:
%%bash
TIMESTAMP=$(date -u +%Y%m%d_%H%M%S)
BASE_OUTPUT_DIR=gs://${BUCKET}/babyweight/hp_tuning_$TIMESTAMP
JOB_NAME=babyweight_hpt_$TIMESTAMP

PYTHON_PACKAGE_URI=gs://${BUCKET}/babyweight/babyweight_trainer-0.1.tar.gz
PYTHON_PACKAGE_EXECUTOR_IMAGE_URI="us-docker.pkg.dev/vertex-ai/training/tf-cpu.2-8:latest"
PYTHON_MODULE=trainer.task

echo > ./hyperparam.yaml "displayName: $JOB_NAME
studySpec:
  metrics:
  - metricId: val_rmse
    goal: MINIMIZE
  parameters:
  - parameterId: batch_size
    integerValueSpec:
      minValue: 8
      maxValue: 512
    scaleType: UNIT_LOG_SCALE
  - parameterId: nembeds
    integerValueSpec:
      minValue: 3
      maxValue: 30
    scaleType: UNIT_LINEAR_SCALE
  algorithm: ALGORITHM_UNSPECIFIED # results in Bayesian optimization
trialJobSpec:
  baseOutputDirectory:
    outputUriPrefix: $BASE_OUTPUT_DIR
  workerPoolSpecs:
  - machineSpec:
      machineType: n1-standard-8
    pythonPackageSpec:
      executorImageUri: $PYTHON_PACKAGE_EXECUTOR_IMAGE_URI
      packageUris:
      - $PYTHON_PACKAGE_URI
      pythonModule: $PYTHON_MODULE
      args:
      - --train_data_path=gs://${BUCKET}/babyweight/data/train*.csv
      - --eval_data_path=gs://${BUCKET}/babyweight/data/eval*.csv
      - --num_epochs=10
      - --train_examples=5000
      - --eval_steps=100
      - --batch_size=32
      - --nembeds=8
    replicaCount: 1"
        
gcloud ai hp-tuning-jobs create \
    --region=$REGION \
    --display-name=$JOB_NAME \
    --config=hyperparam.yaml \
    --max-trial-count=20 \
    --parallel-trial-count=5

## トレーニングの再実行

今回は `batch_size` と `nembeds` に調整済みのパラメータを使用します。最良の結果は以下と異なる場合があるため、ご自身の結果を必ず記入してください！

In [ ]:
%%bash
TIMESTAMP=$(date -u +%Y%m%d_%H%M%S)
OUTDIR=gs://${BUCKET}/babyweight/tuned_$TIMESTAMP
JOB_NAME=babyweight_tuned_$TIMESTAMP

PYTHON_PACKAGE_URI=gs://${BUCKET}/babyweight/babyweight_trainer-0.1.tar.gz
PYTHON_PACKAGE_EXECUTOR_IMAGE_URI="us-docker.pkg.dev/vertex-ai/training/tf-cpu.2-8:latest"
PYTHON_MODULE=trainer.task

echo > ./tuned_config.yaml "workerPoolSpecs:
  machineSpec:
    machineType: n1-standard-8
  replicaCount: 1
  pythonPackageSpec:
    executorImageUri: $PYTHON_PACKAGE_EXECUTOR_IMAGE_URI
    packageUris: $PYTHON_PACKAGE_URI
    pythonModule: $PYTHON_MODULE
    args:
    - --train_data_path=gs://${BUCKET}/babyweight/data/train*.csv
    - --eval_data_path=gs://${BUCKET}/babyweight/data/eval*.csv
    - --output_dir=$OUTDIR
    - --num_epochs=10
    - --train_examples=20000
    - --eval_steps=100
    - --batch_size=32
    - --nembeds=8"
    
gcloud ai custom-jobs create \
  --region=${REGION} \
  --display-name=$JOB_NAME \
  --config=tuned_config.yaml

## ラボのまとめ: 
このラボでは、環境のセットアップ、ハイパーパラメータ引数解析コードを保持するトレーナー モジュールの task.py の作成、Keras モデルコードを保持するトレーナー モジュールの model.py の作成、トレーナー モジュール パッケージのローカルでの実行、および Vertex AI へのトレーニング ジョブの送信を行いました。

Copyright 2021 Google LLC
Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at
    https://www.apache.org/licenses/LICENSE-2.0
Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.